# Applying LangChain to 10Q Reports
_Author_: https://github.com/raffysantayana

## LangChain and LangSmith Constants
Redefining constants just like we did in notebook 03_LangChainandLangSmithPractice.

## Imports

In [1]:
import os
import getpass
import requests
import pandas as pd
from uuid import uuid4
from langsmith import traceable
from langsmith.wrappers import wrap_openai
from datetime import datetime
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import HumanMessage, SystemMessage

# Generate unique ID to help differentiate LangSmith logs
unique_id = uuid4().hex[0:8]
now = str(datetime.today()).replace(" ", "_").replace(":", "_")
with open(f"../langsmith_log_refs/summary_{now}.txt", "w+") as log:
    log.write(f"Unique ID for runs starting at {now}: {unique_id}")
    now = None

# Return to troubleshooting below error when more comfortable with LangChain
# due to OpenAI token cost
# Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')
# Configuring the environment
# os.environ["LANGCHAIN_TRACING_V2"] = "true"
# os.environ["LANGCHAIN_PROJECT"] = "sec10qs"
# os.environ["LANGCHAIN_ENDPOINT"] = r"https://api.smith.langchain.com"
# os.environ["LANGSMITH_WORKSPACE_ID"] = "Workspace 1"

# with open(f"../../keys/langsmith_api_key.txt", "r") as langsmith_key_reader:
    # os.environ["LANGSMITH_API_KEY"] = langsmith_key_reader.read().strip()
    # os.environ["LANGSMITH_API_KEY"] = getpass.getpass("Enter your LangSmith API Key: ")
    
with open(f"../../keys/openai_project_key.txt", "r") as openai_key_reader:
    os.environ["OPENAI_API_KEY"] = openai_key_reader.read().strip()
    # os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API Key: ")

## Creating the Chain
### Templates
The purpose of this repository is to practice prompting, LangChain, LangSmith, and RAG pipeline development. I will overlook cleaning the HTML content of each report for now with plans to return to it. I will prompt the LLM to be an expert as a investment analyst to think from that lens despite LLMs not being able to provide true financial advice. Even thought it is prompted to believe it is an investment analyst, definitions and a general methodology of analysis will be provided.

I will also prompt it to think it is an expert front-end engineer to hopefully help it understand the HTML. Now I'm curious if using the same prompt without the `"You also have extensive experience as a front-end developer, so you are able to parse HTML and undesrtand it's contents."` line will provide significantly different results.

What these prompts do not include are user input to provide further context to the request. I believe the system template provides enough context for the pipeline to provide a summary. I currently do not see any reason why providing no user input will negatively impact the results.

Template 1

```python3
system_template = '''
You are a fiduciary based in New York, NY who has an extensive background as a investment analyst. You understand that 10Q reports are a quarterly updated report due 45 days after the first three quarters of the fiscal year. You also understand that these reports contain unaudited financial statements and provide updates on the company's operations and any significant changes in risk factors.

Key sections of a 10Q form include Financial Statements, Management's Discussion and Analysis (MD&A), Quantitative and Qualitative Disclosures, Legal Proceedings & Risk Updates.
Financial Statements in this context contain unaudited financial statements which show the company's quarterly performance and include comparisons to previous periods.
Management's Discussion and Analysis (MD&A) in this context can be described as management's analysis of the quarterly results, highlighting important trends and explaining significant changes.
Quantitative and Qualitative Disclosures is about market risk which covers how the company manages various financial risks which included but isn't limited to interest rate changes or currency fluctuations.
Legal Proceedings & Risk Updates covers any updates to legal proceedings and risk factors that have emerged since the last annual report.

When approaching these filings, start by reading the MD&A section which provides valuable context about the company's performance and management's perspective on key trends. Next, carefully review financial statements, focusing on the incoming statement, balance sheet, and cash flow statement to understand the compnay's profitability, financial position, and cash generation. Then examinee the risk factor section to understand potential challenges and note how these risks have evolved over time. Finally, review the footnotes of the financial statements. While they might seem technical these notes contain critical details about accounting policies, assumptions, and potential issues that could significantly impact your understanding of the company's true financial condition.

You also have extensive experience as a front-end developer, so you are able to parse HTML and undesrtand it's contents.

Provide a summary report's HTML content is: {content}
'''
```

In [2]:
system_template = '''
You are a fiduciary based in New York, NY who has an extensive background as a investment analyst. You understand that 10Q reports are a quarterly updated report due 45 days after the first three quarters of the fiscal year. You also understand that these reports contain unaudited financial statements and provide updates on the company's operations and any significant changes in risk factors.

Key sections of a 10Q form include Financial Statements, Management's Discussion and Analysis (MD&A), Quantitative and Qualitative Disclosures, Legal Proceedings & Risk Updates.
Financial Statements in this context contain unaudited financial statements which show the company's quarterly performance and include comparisons to previous periods.
Management's Discussion and Analysis (MD&A) in this context can be described as management's analysis of the quarterly results, highlighting important trends and explaining significant changes.
Quantitative and Qualitative Disclosures is about market risk which covers how the company manages various financial risks which included but isn't limited to interest rate changes or currency fluctuations.
Legal Proceedings & Risk Updates covers any updates to legal proceedings and risk factors that have emerged since the last annual report.

You also have extensive experience as a front-end developer, so you are able to parse HTML and undesrtand it's contents.

Provide a summary for each of the 10Q report's sections - Financial Statements, MD&A, Quantitative and Qualitative Disclosures, and Legal Proceedings & Risk Updates.: {content}
'''

Getting a token count to consider GPT-4's 10,000 token limit.

In [3]:
token_count = len(system_template.split())
token_count

229

In [4]:
prompt_template = ChatPromptTemplate.from_messages(
    [
        ("system", system_template)
        # no user input for this prompt, so no user key
        # ("user", "{text}"),
    ]
)

### The Data Retrieved
Retrieving the URLs containing the reports. Will summarize only the first URL.

In [5]:
df_10qs = pd.read_csv('../data/all_10qs.csv')
# Dropping the original indices of the dataframe, which are the default sequence of 0 through N anyways
df_10qs.drop('Unnamed: 0', axis=1, inplace=True)
df_10qs.head()['report_url']

C:\Users\raffy\AppData\Local\Temp\ipykernel_21064\1550298639.py:1: DtypeWarning: Columns (0: film_number) have mixed types. Specify dtype option on import or set low_memory=False.
  df_10qs = pd.read_csv('../data/all_10qs.csv')


0    https://www.sec.gov/Archives/edgar/data/112737...
1    https://www.sec.gov/Archives/edgar/data/112737...
2    https://www.sec.gov/Archives/edgar/data/112737...
3    https://www.sec.gov/Archives/edgar/data/112737...
4    https://www.sec.gov/Archives/edgar/data/112737...
Name: report_url, dtype: str

In [6]:
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36',
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8',
    'Accept-Language': 'en-US,en;q=0.5',
    'Accept-Encoding': 'gzip, deflate',
    'Connection': 'keep-alive',
}
response = requests.get(df_10qs.iloc[0]['report_url'], headers=headers)

if (response.status_code == 200) and (response.content):
    print('Good response')
else:
    print(f'Status Code: {response.status_code}\tResponse Content: {response.content[:20]}')

Good response


### The Model and Parser

We will define our GPT-4 model and StrOutputPraser. The StrOutputParser is used at the end of a chain to parse the outputs from AIMessage objects to strings. We will summarize [this report](https://www.sec.gov/Archives/edgar/data/1127371/000162828024023222/cvcy-20240331.htm#iba7653e7fb8341e984e5ba0c9a21f4e7_76).

In [7]:
model = ChatOpenAI(model="gpt-4")
parser = StrOutputParser()

In [8]:
chain = prompt_template | model | parser
chain.invoke(
    {'content': response.content[:10000-token_count]}
)

"The HTML 10Q report summary provided gives insights in four main areas. These summaries are:\n\n1. Financial Statements: This section provides unaudited financial statements that offer insights into the company's quarterly performance. For instance, it shows that the company has made a loan and lease receivable allowance of 14,658 USD for the current quarter, compared to 14,653 USD for the same period in the previous year. It also reveals that the company has approximately 11,831,994 outstanding common stock shares.\n\n2. Management's Discussion and Analysis (MD&A): This section is expected to highlight important trends and explain significant changes but is not present in the provided HTML code.\n\n3. Quantitative and Qualitative Disclosures: This section would typically cover how the company manages various financial risks, such as interest rate changes or currency fluctuations. However, it cannot be summarized from the given XML as those elements are not present.\n\n4. Legal Procee

## Findings and Limitations

It seems the model incorrectly stated that "the company has made a loan and lease receivable allowance of 14,658 USD for the current quarter, compared to 14,653 USD for the same period in the previous year." where the "current" quarter is March 31, 2024. The quarter being compared to March 31, 2024 is actually December 31, 2023, not March 31, 2023.

Due to the 10,000 token limit for GPT-4, it seems that a large chunk of the HTML content has been trimmed off. It seems the model was only able to read the Cautionary Note Regarding Forward-Looking Statements and a portion of the first figure in Part 1: Financial Information, Item 1: Financial Statements, Community WEst Bancshares and Subsidiary Consolidated Balance Sheets (Unaudited). This caused the model to find no information regarding MD&A, Quantitative and Qualitative Disclosures, and Legal Proceedings & Risk Updates.

## Next Steps
Considering the 10,000 token limit, it may be beneficial to actually parse the HTML prior to passing into the chain and request 4 summaries for each Financial Statement section, MD&A section, Quantitative and Qualitatve Disclosure Section, and Legal Proceedings & Risk Updates section. It would be interesting to use the four output and get one last summary of all four of them in one output which reminds me of MapReduce in some way.

This also sounds more like multi-query prompting, but isn't exactly it. Multi-query takes a user's original question like "What is principal component analysis?" and a system template will prompt the LLM to create n-number of ways to rewrite the question. By creating multiple perspectives of the same question provided by the user, this will hopefully provide the retrieval step better context of what documents to retrieve for the RAG pipelines output. I believe this to be a similar principle to ensemble learning where multiple "weak" models working together as one model providing more accurate results. While I am not using distance-based similarity search to retrieve documents as part of the pipeline, I am just noticing some similaries to query translation options to this repo's future steps.